In [5]:
WAREHOUSE  = "analytics_warehouse"
SCHEMA     = "gold"
TARGET     = f"{WAREHOUSE}.{SCHEMA}.FactAirQualityDaily"
write_mode = "overwrite"

AQI_BREAKPOINTS = [
    (0.0,   12.0,  "Good"),
    (12.1,  35.4,  "Moderate"),
    (35.5,  55.4,  "Unhealthy for Sensitive Groups"),
    (55.5,  150.4, "Unhealthy"),
    (150.5, 250.4, "Very Unhealthy"),
    (250.5, 9999,  "Hazardous"),
]

GOLD_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/gold.Lakehouse"
)

SILVER_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/silver.Lakehouse"
)

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 7, Finished, Available, Finished, False)

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.types import (
    IntegerType, LongType, FloatType, StringType, BooleanType
)

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 12, Finished, Available, Finished, False)

In [7]:
aq_df = spark.read.format("delta").load(f"{SILVER_BASE}/Tables/dbo/air_quality_silver")
raw_count = aq_df.count()
print(f"Silver air_quality rows : {raw_count:,}")
aq_df.printSchema()

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 9, Finished, Available, Finished, False)

Silver air_quality rows : 49,812
root
 |-- location_id: long (nullable = true)
 |-- location_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- measured_date: date (nullable = true)
 |-- avg_pm25: double (nullable = true)
 |-- avg_no2: double (nullable = true)
 |-- avg_o3: double (nullable = true)
 |-- location_key: string (nullable = true)



In [11]:
fact_df = (
    aq_df
    .withColumn("date_key",
        F.date_format("measured_date", "yyyyMMdd").cast(IntegerType()))
    .withColumn("location_key",
        F.sha2(
            F.concat_ws("|",
                F.col("location_id").cast(StringType()),
                F.coalesce(F.col("location_name"), F.lit(""))
            ), 256
        )
    )
    .withColumn("avg_pm25", F.col("avg_pm25").cast(FloatType()))
    .withColumn("avg_no2",  F.col("avg_no2").cast(FloatType()))
    .withColumn("avg_o3",   F.col("avg_o3").cast(FloatType()))
    .withColumn("aqi_category", aqi_expr)
    .select(
        "date_key",
        "location_key",
        "avg_pm25",
        "avg_no2",
        "avg_o3",
        "aqi_category",
    )
    .dropDuplicates(["date_key", "location_key"])
)

fact_count = fact_df.count()
print(f"\nFact rows to write : {fact_count:,}")

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 13, Finished, Available, Finished, False)


Fact rows to write : 49,812


In [12]:
FACT_AQ_PATH = f"{GOLD_BASE}/Tables/dbo/factairqualitydaily"

(
    fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FACT_AQ_PATH)
)
print(f"[OK] FactAirQualityDaily written  ({fact_df.count():,} rows)")

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 14, Finished, Available, Finished, False)

[OK] FactAirQualityDaily written  (49,812 rows)


In [13]:
val_df = spark.read.format("delta").load(FACT_AQ_PATH)

print("\n--- FactAirQualityDaily summary ---")
val_df.agg(
    F.min("date_key").alias("earliest_date_key"),
    F.max("date_key").alias("latest_date_key"),
    F.round(F.avg("avg_pm25"), 2).alias("overall_avg_pm25"),
    F.round(F.avg("avg_no2"),  2).alias("overall_avg_no2"),
    F.round(F.avg("avg_o3"),   2).alias("overall_avg_o3"),
    F.count("*").alias("fact_rows"),
).show()

print("--- AQI category distribution ---")
(
    val_df
    .groupBy("aqi_category")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .show()
)

StatementMeta(, 3f1dd823-eb0a-4f6c-affc-1807e5aff859, 15, Finished, Available, Finished, True)


--- FactAirQualityDaily summary ---
+-----------------+---------------+----------------+---------------+--------------+---------+
|earliest_date_key|latest_date_key|overall_avg_pm25|overall_avg_no2|overall_avg_o3|fact_rows|
+-----------------+---------------+----------------+---------------+--------------+---------+
|         20160306|       20260521|            8.28|           0.02|          0.03|    49812|
+-----------------+---------------+----------------+---------------+--------------+---------+

--- AQI category distribution ---
+--------------------+-----+
|        aqi_category|count|
+--------------------+-----+
|                Good|39442|
|            Moderate| 8090|
|             Unknown| 2052|
|Unhealthy for Sen...|  171|
|           Unhealthy|   50|
|      Very Unhealthy|    7|
+--------------------+-----+

